# Benchmark Analysis

Loads `benchmark_results.csv`, summarizes metrics, and plots comparisons.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style="whitegrid")
DATA_PATH = Path('benchmark_results.csv')

In [ ]:
df = pd.read_csv(DATA_PATH)
numeric = ['parameters','estimated_size_bytes','average_latency_ms','accuracy','macro_f1','weighted_f1','top_3_accuracy']
for c in numeric:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
df

In [ ]:
# Summary and best-by-metric
summary = df.describe(include='all')
best = {}
if 'parameters' in df.columns:
    best['fewest_parameters'] = df.loc[df['parameters'].idxmin()].to_dict()
if 'estimated_size_bytes' in df.columns:
    best['smallest_model'] = df.loc[df['estimated_size_bytes'].idxmin()].to_dict()
if 'average_latency_ms' in df.columns:
    best['fastest_model'] = df.loc[df['average_latency_ms'].idxmin()].to_dict()
for metric in ['accuracy','macro_f1','weighted_f1','top_3_accuracy']:
    if metric in df.columns:
        best[f'best_{metric}'] = df.loc[df[metric].idxmax()].to_dict()
summary, best

In [ ]:
import os
out_dir = Path('notebooks/figures')
out_dir.mkdir(parents=True, exist_ok=True)

def plot_bar(col, title, ylabel, log=False):
    plt.figure(figsize=(8,4))
    df_sort = df.sort_values(col, ascending=False)
    ax = sns.barplot(x='model_name', y=col, data=df_sort, palette='muted')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    if log:
        ax.set_yscale('log')
    plt.xticks(rotation=45)
    plt.tight_layout()
    path = out_dir / f'{col}.png'
    plt.savefig(path)
    print('Saved', path)

if 'parameters' in df.columns:
    plot_bar('parameters','Model Parameter Count','Parameters', log=True)
if 'estimated_size_bytes' in df.columns:
    plot_bar('estimated_size_bytes','Estimated Model Size (bytes)','Bytes', log=True)
if 'average_latency_ms' in df.columns:
    plot_bar('average_latency_ms','Average Inference Latency (ms)','ms', log=False)

metrics = [m for m in ['accuracy','macro_f1','weighted_f1','top_3_accuracy'] if m in df.columns]
if metrics:
    plt.figure(figsize=(10,5))
    melted = df[['model_name']+metrics].melt(id_vars='model_name', value_vars=metrics, var_name='metric', value_name='value')
    ax = sns.barplot(x='model_name', y='value', hue='metric', data=melted)
    ax.set_title('Evaluation metrics by model')
    plt.xticks(rotation=45)
    plt.legend(title='Metric')
    plt.tight_layout()
    path = out_dir / 'metrics.png'
    plt.savefig(path)
    print('Saved', path)

### How to run benchmarks (Windows)

Set `PYTHONPATH` and run the benchmark script, then analyze results:

```powershell
set PYTHONPATH=.
python scripts/run_benchmarks.py --validation-dir path\to\val --output benchmark_results.csv
python notebooks/benchmark_analysis.py --input benchmark_results.csv --show
```